In [24]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [25]:
df=pd.read_csv("train.csv")

In [26]:
df.head()

,id,carat,cut,color,clarity,depth,table,x,y,z,price
0,0,1.52,Premium,F,VS2,62.2,58.0,7.27,7.33,4.55,13619
1,1,2.03,Very Good,J,SI2,62.0,58.0,8.06,8.12,5.05,13387
2,2,0.70,Ideal,G,VS1,61.2,57.0,5.69,5.73,3.50,2772
3,3,0.32,Ideal,G,VS1,61.6,56.0,4.38,4.41,2.71,666
4,4,1.70,Premium,G,VS2,62.6,59.0,7.65,7.61,4.77,14453


In [27]:
df = df.drop(labels=['id'],axis=1)

In [28]:
X = df.drop(labels=['price'],axis=1)
Y = df[['price']]

In [29]:
# Define which columns should be ordinal-encoded and which should be scaled
categorical_cols = X.select_dtypes(include='object').columns
numerical_cols = X.select_dtypes(exclude='object').columns
            
# Define the custom ranking for each ordinal variable
cut_categories = ['Fair', 'Good', 'Very Good','Premium','Ideal']
color_categories = ["J","I","H","G","F","E","D"]
clarity_categories = ['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF']

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder,StandardScaler

# Numerical Pipeline
num_pipeline = Pipeline(
                steps = [
                ('imputer',SimpleImputer(strategy='median')),
                ('scaler',StandardScaler())                
                ]
            )

# Categorical Pipeline
cat_pipeline = Pipeline(
                steps=[
                ('imputer',SimpleImputer(strategy='most_frequent')),
                ('ordinal_encoder',OrdinalEncoder(categories=[cut_categories,color_categories,clarity_categories])),
                ('scaler',StandardScaler())
                ]
            )

preprocessor = ColumnTransformer(
                [
                ('num_pipeline',num_pipeline,numerical_cols),
                ('cat_pipeline',cat_pipeline,categorical_cols)
                ]
            )

In [30]:
from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(X,Y,test_size=0.2,random_state=42)

In [31]:
xtrain = pd.DataFrame(preprocessor.fit_transform(xtrain),columns=preprocessor.get_feature_names_out())
xtest = pd.DataFrame(preprocessor.transform(xtest),columns=preprocessor.get_feature_names_out())

In [32]:
preprocessor.get_feature_names_out()

array(['num_pipeline__carat', 'num_pipeline__depth',
       'num_pipeline__table', 'num_pipeline__x', 'num_pipeline__y',
       'num_pipeline__z', 'cat_pipeline__cut', 'cat_pipeline__color',
       'cat_pipeline__clarity'], dtype=object)

In [33]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "K-Neighbors Regressor": KNeighborsRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest Regressor": RandomForestRegressor(),
    "XGB Regressor": XGBRegressor(),
    "CatBoost Regressor": CatBoostRegressor(verbose=0) 
}

# Function to evaluate each model
def evaluate_model(true, predicted):
    mae = mean_absolute_error(true, predicted)
    mse = mean_squared_error(true, predicted)
    rmse = np.sqrt(mse)
    r2 = r2_score(true, predicted)
    return mae, rmse, r2

# Dictionary to store results
results = {}

# Train and evaluate each model
for name, model in models.items():
    model.fit(xtrain, ytrain)          # Train model
    y_pred = model.predict(xtest)      # Predict
    mae, rmse, r2 = evaluate_model(ytest, y_pred)
    results[name] = [mae, rmse, r2]
    print(f"✅ {name} done")

# Convert results to a dataframe for easy comparison
results_df = pd.DataFrame(results, index=['MAE', 'RMSE', 'R2']).T
print("\n📊 Model Evaluation Results:\n")
print(results_df.sort_values(by='R2', ascending=False))

✅ Linear Regression done
✅ Lasso done
✅ Ridge done
✅ K-Neighbors Regressor done
✅ Decision Tree done


d:\STUDY CODING\euphoria\Gemstone Price Prediction\gemenv\Lib\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


✅ Random Forest Regressor done
✅ XGB Regressor done
✅ CatBoost Regressor done

📊 Model Evaluation Results:

                                MAE         RMSE        R2
CatBoost Regressor       294.647370   578.508608  0.979290
XGB Regressor            296.983582   585.518894  0.978785
Random Forest Regressor  309.062423   605.762314  0.977292
K-Neighbors Regressor    349.467968   671.287866  0.972114
Decision Tree            422.166137   831.617581  0.957203
Linear Regression        671.585639  1006.600986  0.937298
Ridge                    671.613741  1006.606231  0.937297
Lasso                    672.863489  1006.871581  0.937264


In [34]:
# Extract only R2 scores from the results dictionary
r2_scores = {model: metrics[2] for model, metrics in results.items()}

# Convert to DataFrame
df_results = pd.DataFrame(list(r2_scores.items()), columns=['Model Name', 'R2_Score'])

# Sort descending by R2
df_results = df_results.sort_values(by='R2_Score', ascending=False).reset_index(drop=True)

# Display
df_results


,Model Name,R2_Score
0,CatBoost Regressor,0.979290
1,XGB Regressor,0.978785
2,Random Forest Regressor,0.977292
3,K-Neighbors Regressor,0.972114
4,Decision Tree,0.957203
5,Linear Regression,0.937298
6,Ridge,0.937297
7,Lasso,0.937264


In [ ]:
final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", CatBoostRegressor(verbose=0))
])

# Train
final_pipeline.fit(X,Y)

# Save
joblib.dump(final_pipeline, "1_final_model.pkl")
print("✅ Saved full model pipeline as final_model.pkl")


✅ Saved full model pipeline as final_model.pkl
